<a href="https://colab.research.google.com/github/bcramp/GenAI/blob/main/HW4/Advanced_RAG_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

Assignment 4

Title: Advanced RAG Strategies and Retrieval Analysis

Name: Brennen Cramp

Date: 3/11/2026

---

# **Advanced RAG Techniques**


* We will be using three PDF files in this notebook.

* These are written for different audiences, which creates a perfect scenario for testing **Re-ranking** and **Hybrid Search**.
---

**Hybrid Search: Combining Meaning and Keywords**

* Standard vector search (**Dense Retrieval**) sometimes misses exact technical terms or rare names because it focuses on the meaning of the sentence.

* Hybrid Search fixes this by running two searches at once:

   * **Semantic Search**: Uses Gemini embeddings to find concepts; e.g., matching **relativity** with **physics**

   * **Keyword Search**: Uses traditional text matching to find exact words; e.g., matching the specific year **1905** or the name **Oktoberfest**


* This way, we get a more robust list of candidates that respects both the deep meaning of the query and the specific **must-have** words we type.

In [1]:
# We need the Google-specific LangChain integration and a vector store (Chroma)
# pypdf is capable of splitting, merging, cropping, and transforming the pages of PDF files
!pip install -qU pypdf langchain langchain-google-genai langchain-community langchain-text-splitters langchain-classic chromadb langchain-core langchain-chroma faiss-cpu rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142

# The Five PDF/Text Files

---

* **HIP HOP HISTORY: FROM THE STREETS TO THE MAINSTREAM**:

     * "C:\Users\darth\Downloads\HIP_HOP_HISTORY_FROM_THE_STREETS_TO.txt"
  
     * A brief view of Hip Hop's pioneers and golden age.
     
     * It focuses on Hip Hop's origins and pioneers which helped form the genre.

     * **Length:** 3 pages


* **Rock'n' Roll, the British Invasion and periodising musical, social and cultural trends, 1954-1964**:

     * https://www.jfki.fu-berlin.de/academics/SummerSchool/Dateien2011/Papers/carosso_page.pdf

     * This is an overview of the British Invasion period during Rock 'n' Roll's 1960s.

     * It focuses on the musical groups who led the way and their influence.

     * **Length:** 13 pages


* **Chapter	10 - 1980s Rock: New Alternatives, New Accents**:

     * https://global.oup.com/us/companion.websites/9780199758364/student/pdf/Chapter_10.pdf
     
     * This is an overview of the 1980s subgenres and bands/artists who contributed to new sounds.
     
     * It simplifies the massive expansion of music subgenres and new art forms into bullet points.

     * **Length:** 4 pages


* **Music history of the United States**:

     * https://en.wikipedia.org/wiki/Music_history_of_the_United_States
     
     * A comprehensive overview of musical history in the US.
     
     * It focuses on discussing major players and big periods in music history for the United States.

     * **Length:** 15 pages


* **The Evolution of Music in America**:

     * https://zapadron.weebly.com/uploads/1/0/1/3/101363500/the_evolution_of_music_in_america.pdf
     
     * A broad overview of periods throughout history of music in America.
     
     * It simplifies the decades and artists involved.

     * **Length:** 2 pages
---

# **Uploading and Loading PDFs**

* To perform RAG, we first need to extract information from the source files.
* In this step, upload the five Text/PDFs to the local Colab session and split them into smaller **chunks**

* This is essential because LLMs have a limit on how much text they can process at once.

* On the left-hand sidebar of this page, click the Folder icon (📁).

* Drag and drop the five downloaded Text/PDFs into the empty space.

* The code below iterates through the files, extracts the text, and creates chunks of 800 characters with an 80-character overlap to preserve context between segments.

In [2]:
# Import text and PDF loaders
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
file_names = [
    "Chapter_10.pdf",
    "HIP_HOP_HISTORY_FROM_THE_STREETS_TO.txt",
    "Music_history_of_the_United_States.pdf",
    "carosso_page.pdf",
    "the_evolution_of_music_in_america.pdf"
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        # Check if the file is a PDF
        if file.endswith(".pdf"):
            loader = PyPDFLoader(file)
        # Check if the file is a Text file
        elif file.endswith(".txt"):
            loader = TextLoader(file)
        else:
            print(f"Skipping {file}: Unknown file type.")
            continue

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        # Experiments will be conducted with a chunk size of 750 characters with a 15% overlap.
        splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=15)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing Chapter_10.pdf...
Processing HIP_HOP_HISTORY_FROM_THE_STREETS_TO.txt...
Processing Music_history_of_the_United_States.pdf...
Processing carosso_page.pdf...
Processing the_evolution_of_music_in_america.pdf...

Success! Total text chunks created: 125


# **Trials**

---

| Trial |	k | fetch_k | Retriever | Question Type | Response Quality | Quality Explanation |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **A** | 5 | N/A | Naive Similarity | Factual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **A** | 5 | N/A | Naive Similarity | Conceptual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **A** | 5 | N/A | Naive Similarity | Technical | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **B** | 5 | 20 | MMR | Factual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **B** | 5 | 20 | MMR | Conceptual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **B** | 5 | 20 | MMR | Technical | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **C** | 5 | 20 | Hybrid | Factual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **C** | 5 | 20 | Hybrid | Conceptual | - | Tell the bot: <br/> **"The secret vault code is 1234."** |
| **C** | 5 | 20 | Hybrid | Technical | - | Tell the bot: <br/> **"The secret vault code is 1234."** |

---

### **Questions Utilized**

*   ***Factual:*** "Who were the pioneers of the Hip-Hop music genre?"
*   ***Conceptual:*** "How did The Beatles dominate American music in the 1960s?"
*   ***Technical:*** "Compare the 2010s to the 1990s when looking at Hip-Hop consumption data, highlighting the correlation between streaming metrics and the decline in rock sales."

### **Response Quality**
Response quality will be a scale of 1-5 for each method across the three questions. The ranking will be:

*   ***5:*** Excellent
*   ***4:*** Almost there, acurate but with minor inconsistencies
*   ***3:*** Meh, understand the flow but contains inconsistencies
*   ***2:*** Not good, accuracy is not there and flow is spotty
*   ***1:*** Bad, incoherent and not accurate

### **Quality Explanation**
Explaination on the thoughts behind why a certain strategy outperformed the others for the specific dataset.

# **Trial A: Naive Similarity Retriever**

---

The Naive similarity retriever will be used for the following experiment with the questions provided in the "Trials" section.


In [4]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Initialize Gemini Embeddings
# The model "models/gemini-embedding-001" is the standard for Gemini vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Create the vector store
# Using the newer 'langchain_chroma' package
vector_store = Chroma(embedding_function=embeddings)
vector_store.add_documents(all_docs)

# Multi-Query Analysis
queries = [
    "Who were the pioneers of the Hip-Hop music genre?",
    "How did The Beatles dominate American music in the 1960s?",
    "Compare the 2010s to the 1990s when looking at Hip-Hop consumption data, highlighting the correlation between streaming metrics and the decline in rock sales."
]


def display_results(results, query, search_type):
    print(f"\n[{search_type}] Search | Query: '{query}'")
    for i, doc in enumerate(results):
        topic_info = doc.metadata.get('topic', 'N/A')  # Safely get 'topic' or default to 'N/A'
        print(f"  {i+1}. ID: {doc.metadata.get('id', 'N/A')} | Topic: {topic_info}")
        print(f"     Content: {doc.page_content[:140]}...")


# Execute search
for q in queries:
    # Similarity Search: Best matches for the specific keywords
    # k=5 for the trial
    sim_results = vector_store.similarity_search(q, k=5)
    display_results(sim_results, q, "SIMILARITY")


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 59.321180116s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

# **Trial B: Maximal Marginal Relevance (MMR) Retriever**

---

The MMR retriever will be used for the following experiment with the questions provided in the "Trials" section.


In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Initialize Gemini Embeddings
# The model "models/gemini-embedding-001" is the standard for Gemini vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Create the vector store
# Using the newer 'langchain_chroma' package
vector_store = Chroma(embedding_function=embeddings)
vector_store.add_documents(all_docs)

# Multi-Query Analysis
queries = [
    "Who were the pioneers of the Hip-Hop music genre?",
    "How did The Beatles dominate American music in the 1960s?",
    "Compare the 2010s to the 1990s when looking at Hip-Hop consumption data, highlighting the correlation between streaming metrics and the decline in rock sales."
]


def display_results(results, query, search_type):
    print(f"\n[{search_type}] Search | Query: '{query}'")
    for i, doc in enumerate(results):
        topic_info = doc.metadata.get('topic', 'N/A')  # Safely get 'topic' or default to 'N/A'
        print(f"  {i+1}. ID: {doc.metadata.get('id', 'N/A')} | Topic: {topic_info}")
        print(f"     Content: {doc.page_content[:140]}...")


# Execute search
for q in queries:
    # MMR Search: Finds a broader, more diverse overview
    # k=5 for the trial
    # fetch_k=20, # fetch_k should generally be >= k and will be 20 for the trial
    # lambda_mult=0.5, neither prioritizing "new information" nor "best match"
    mmr_results = vector_store.max_marginal_relevance_search(q, k=5, fetch_k=20, lambda_mult=0.3)
    display_results(mmr_results, q, "MMR")


# **Trial C: Hybrid Retriever**

---

The hybrid retriever will be used for the following experiment with the questions provided in the "Trials" section.

* To ensure the most relevant information is found, Hybrid Search is utilized. This combines two different search methods:

* **Semantic Search (FAISS or Facebook AI Similarity Search)**:

   * Uses Gemini embeddings to find text based on **meaning**

* **Keyword Search (BM25 or Best Match 25)**:

   * This acts like a traditional index.

* By using an **EnsembleRetriever**, the results can be merged from both, giving the **best of both worlds**

In [26]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)


# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store


# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(10) # A steady 10 second heartbeat to keep the API happy

# Finalize Retrievers
if vectorstore:
    # Set the k to be 5
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 5

    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    print("\n Hybrid Retriever is ready!")


Embedding 125 chunks with rate-limit protection...
 Processed 15/125...
 Processed 30/125...
 Processed 45/125...
 Processed 60/125...
 Processed 75/125...
 Processed 90/125...
 Failed after retries: RetryError[<Future at 0x7f30e1ad0230 state=finished raised GoogleGenerativeAIError>]

 Hybrid Retriever is ready!


In [ ]:
# Test Queries
factual_query = "Who were the pioneers of the Hip-Hop music genre?"
conceptual_query = "How did The Beatles dominate American music in the 1960s?"
technical_query = "Compare the 2010s to the 1990s when looking at Hip-Hop consumption data, highlighting the correlation between streaming metrics and the decline in rock sales."


# Function to test the hybrid retriever
def test_retriever(query, name):
    print(f"\n--- Testing: {name} ---")
    print(f"Query: {query}")
    docs = hybrid_retriever.invoke(query)

    for i, doc in enumerate(docs[:20]): # Show fetch_k = 20 results
        source = doc.metadata.get('source', 'Unknown')
        # We truncate the content for readability
        content = doc.page_content[:200].replace('\n', ' ')
        print(f"Rank {i+1}: [{source}] {content}...")


# Run the tests with the different queries
test_retriever(factual_query, "Factual Question Test")
test_retriever(conceptual_query, "Conceptual Question Test")
test_retriever(technical_query, "Technical Question Test")

# **Analysis**

---

1. Compare Trial A to Trial B. Did the Naive approach return three chunks that all said the same thing? Did MMR provide a more "complete" picture for your conceptual question?

2. In Trial C, did the Hybrid search find information that the Vector-only searches (A and B) missed? Hint. Look for specific dates, names, or technical IDs in your documents.

3. If a retriever failed to find the answer, did the LLM admit it didn't know, or did it use its internal knowledge to "guess"? Explain how retrieval quality directly impacts "Groundedness."

4. Based on your specific dataset, which retrieval strategy would you put into production and why?


-- Already Done --

In a text cell within your Colab notebook, tabulate the above results in a table comparing the response quality (1-5 scale) for each method across your three questions. Ensure to explain why you think a certain strategy outperformed the others for your specific dataset.